# Ferretería Horizonte · predicción de ventas y decisiones de inventario
**Perú · precios en soles (S/) · datos SINTÉTICOS · ejecución en CPU**

Cuaderno autónomo: no necesitas subir un CSV, conectar Drive ni configurar claves.
En [Google Colab](https://colab.research.google.com/), elige **Archivo → Subir cuaderno**,
selecciona este `.ipynb` y pulsa **Entorno de ejecución → Ejecutar todas**.
La última celda descarga un ZIP con datos, modelo, métricas, matrices y gráficos.

**Tamaño:** 320 productos × 365 días = **116,800 registros diarios**.
El catálogo procede del proyecto local; las ventas, precios de catálogo y existencias
de ejemplo no son transacciones ni cotizaciones reales de Sodimac, Ace o Toolstation.
El objetivo es estimar **unidades vendidas durante los siguientes 7 días por producto**.
No entrenamos un chatbot ni un modelo de lenguaje.

**Límite del experimento:** las ventas se simulan sin restricciones de disponibilidad.
El stock es un escenario separado para evaluar decisiones, sin recepciones durante
los próximos 7 días. Por ello, las alertas son *riesgo simulado*, no roturas de stock reales.
Las métricas sintéticas comprueban el flujo técnico, no garantizan rendimiento comercial.

## 1. Preparación
Usa las librerías que Google Colab ya incluye: **no instala ni cambia NumPy, pandas,
SciPy o scikit-learn**. No requiere GPU ni servicios de pago. Los datos se generan aquí.

**Si ejecutaste la versión anterior del cuaderno:** selecciona primero
**Entorno de ejecución → Desconectar y eliminar entorno de ejecución**, vuelve a
conectar y ejecuta todas las celdas de esta versión. Esto restablece las librerías
originales de Colab; borra los archivos temporales de esa sesión, no tu cuaderno.
Guarda primero cualquier archivo temporal que quieras conservar.
[Ayuda oficial de Colab](https://research.google.com/colaboratory/faq.html).

In [ ]:
import os, sys, importlib.util
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
requeridas = ['numpy', 'pandas', 'matplotlib', 'sklearn', 'joblib', 'threadpoolctl']
faltantes = [nombre for nombre in requeridas if importlib.util.find_spec(nombre) is None]
if faltantes:
    raise RuntimeError('Faltan librerías incluidas en el entorno estándar de Colab: '
                       + ', '.join(faltantes) + '. Abre este cuaderno en Google Colab.')

import json, zipfile, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score,
    f1_score, accuracy_score, balanced_accuracy_score)
from threadpoolctl import threadpool_limits

SEED = 20261010
DIAS = 365
INICIO = '2025-10-10'
HORIZONTE = 7
SALIDA = Path('resultados_horizonte')
SALIDA.mkdir(exist_ok=True)
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': .2})
print('Preparación lista: se conservan las librerías del entorno, sin instalaciones.')
print('Python:', platform.python_version(), '| NumPy:', np.__version__,
      '| pandas:', pd.__version__, '| sklearn:', sklearn.__version__)

In [ ]:
CATALOGO = [{'sku': 'FER-001-01', 'name': 'Martillo de uña con mango de fibra 16 oz', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-001-02', 'name': 'Martillo de uña con mango de fibra 20 oz', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 4990, 'min_stock': 8}, {'sku': 'FER-001-03', 'name': 'Martillo de uña con mango de fibra 22 oz', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 5990, 'min_stock': 8}, {'sku': 'FER-001-04', 'name': 'Martillo de uña con mango de fibra 25 oz', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 6990, 'min_stock': 8}, {'sku': 'FER-001-05', 'name': 'Martillo de uña con mango de fibra 27 oz', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 7990, 'min_stock': 8}, {'sku': 'FER-002-01', 'name': 'Alicate universal de acero 6 pulgadas', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-002-02', 'name': 'Alicate universal de acero 7 pulgadas', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 3490, 'min_stock': 8}, {'sku': 'FER-002-03', 'name': 'Alicate universal de acero 8 pulgadas', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-002-04', 'name': 'Alicate universal de acero 9 pulgadas', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 4990, 'min_stock': 8}, {'sku': 'FER-002-05', 'name': 'Alicate universal de acero 10 pulgadas', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 5990, 'min_stock': 8}, {'sku': 'FER-003-01', 'name': 'Llave ajustable cromada 6 pulgadas', 'category': 'Herramientas manuales', 'brand': 'Bahco', 'price_cents': 4990, 'min_stock': 8}, {'sku': 'FER-003-02', 'name': 'Llave ajustable cromada 8 pulgadas', 'category': 'Herramientas manuales', 'brand': 'Bahco', 'price_cents': 6490, 'min_stock': 8}, {'sku': 'FER-003-03', 'name': 'Llave ajustable cromada 10 pulgadas', 'category': 'Herramientas manuales', 'brand': 'Bahco', 'price_cents': 7990, 'min_stock': 8}, {'sku': 'FER-003-04', 'name': 'Llave ajustable cromada 12 pulgadas', 'category': 'Herramientas manuales', 'brand': 'Bahco', 'price_cents': 9990, 'min_stock': 8}, {'sku': 'FER-003-05', 'name': 'Llave ajustable cromada 15 pulgadas', 'category': 'Herramientas manuales', 'brand': 'Bahco', 'price_cents': 15990, 'min_stock': 8}, {'sku': 'FER-004-01', 'name': 'Destornillador de punta Phillips PH0 x 75 mm', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 1290, 'min_stock': 8}, {'sku': 'FER-004-02', 'name': 'Destornillador de punta Phillips PH1 x 100 mm', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 1590, 'min_stock': 8}, {'sku': 'FER-004-03', 'name': 'Destornillador de punta Phillips PH2 x 100 mm', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 1790, 'min_stock': 8}, {'sku': 'FER-004-04', 'name': 'Destornillador de punta Phillips PH2 x 150 mm', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-004-05', 'name': 'Destornillador de punta Phillips PH3 x 200 mm', 'category': 'Herramientas manuales', 'brand': 'Stanley', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-005-01', 'name': 'Taladro percutor inalámbrico con maletín 18 V batería 1.5 Ah', 'category': 'Herramientas eléctricas', 'brand': 'Bosch', 'price_cents': 39900, 'min_stock': 8}, {'sku': 'FER-005-02', 'name': 'Taladro percutor inalámbrico con maletín 18 V batería 2 Ah', 'category': 'Herramientas eléctricas', 'brand': 'Bosch', 'price_cents': 45900, 'min_stock': 8}, {'sku': 'FER-005-03', 'name': 'Taladro percutor inalámbrico con maletín 18 V batería 3 Ah', 'category': 'Herramientas eléctricas', 'brand': 'Bosch', 'price_cents': 55900, 'min_stock': 8}, {'sku': 'FER-005-04', 'name': 'Taladro percutor inalámbrico con maletín 18 V batería 4 Ah', 'category': 'Herramientas eléctricas', 'brand': 'Bosch', 'price_cents': 65900, 'min_stock': 8}, {'sku': 'FER-005-05', 'name': 'Taladro percutor inalámbrico con maletín 18 V batería 5 Ah', 'category': 'Herramientas eléctricas', 'brand': 'Bosch', 'price_cents': 75900, 'min_stock': 8}, {'sku': 'FER-006-01', 'name': 'Amoladora angular inalámbrica 18 V 115 mm sin batería', 'category': 'Herramientas eléctricas', 'brand': 'Makita', 'price_cents': 38900, 'min_stock': 8}, {'sku': 'FER-006-02', 'name': 'Amoladora angular inalámbrica 18 V 115 mm batería 2 Ah', 'category': 'Herramientas eléctricas', 'brand': 'Makita', 'price_cents': 54900, 'min_stock': 8}, {'sku': 'FER-006-03', 'name': 'Amoladora angular inalámbrica 18 V 115 mm batería 3 Ah', 'category': 'Herramientas eléctricas', 'brand': 'Makita', 'price_cents': 64900, 'min_stock': 8}, {'sku': 'FER-006-04', 'name': 'Amoladora angular inalámbrica 18 V 125 mm batería 4 Ah', 'category': 'Herramientas eléctricas', 'brand': 'Makita', 'price_cents': 79900, 'min_stock': 8}, {'sku': 'FER-006-05', 'name': 'Amoladora angular inalámbrica 18 V 125 mm batería 5 Ah', 'category': 'Herramientas eléctricas', 'brand': 'Makita', 'price_cents': 89900, 'min_stock': 8}, {'sku': 'FER-007-01', 'name': 'Sierra circular inalámbrica para madera 18 V 165 mm sin batería', 'category': 'Herramientas eléctricas', 'brand': 'DeWalt', 'price_cents': 59900, 'min_stock': 8}, {'sku': 'FER-007-02', 'name': 'Sierra circular inalámbrica para madera 18 V 165 mm batería 2 Ah', 'category': 'Herramientas eléctricas', 'brand': 'DeWalt', 'price_cents': 74900, 'min_stock': 8}, {'sku': 'FER-007-03', 'name': 'Sierra circular inalámbrica para madera 18 V 165 mm batería 4 Ah', 'category': 'Herramientas eléctricas', 'brand': 'DeWalt', 'price_cents': 89900, 'min_stock': 8}, {'sku': 'FER-007-04', 'name': 'Sierra circular inalámbrica para madera 18 V 184 mm batería 4 Ah', 'category': 'Herramientas eléctricas', 'brand': 'DeWalt', 'price_cents': 99900, 'min_stock': 8}, {'sku': 'FER-007-05', 'name': 'Sierra circular inalámbrica para madera 18 V 184 mm batería 5 Ah', 'category': 'Herramientas eléctricas', 'brand': 'DeWalt', 'price_cents': 119900, 'min_stock': 8}, {'sku': 'FER-008-01', 'name': 'Lijadora rotoorbital inalámbrica 18 V 125 mm sin batería', 'category': 'Herramientas eléctricas', 'brand': 'Makita', 'price_cents': 39900, 'min_stock': 8}, {'sku': 'FER-008-02', 'name': 'Lijadora rotoorbital inalámbrica 18 V 125 mm batería 2 Ah', 'category': 'Herramientas eléctricas', 'brand': 'Makita', 'price_cents': 55900, 'min_stock': 8}, {'sku': 'FER-008-03', 'name': 'Lijadora rotoorbital inalámbrica 18 V 125 mm batería 3 Ah', 'category': 'Herramientas eléctricas', 'brand': 'Makita', 'price_cents': 62900, 'min_stock': 8}, {'sku': 'FER-008-04', 'name': 'Lijadora rotoorbital inalámbrica 18 V 125 mm batería 4 Ah', 'category': 'Herramientas eléctricas', 'brand': 'Makita', 'price_cents': 69900, 'min_stock': 8}, {'sku': 'FER-008-05', 'name': 'Lijadora rotoorbital inalámbrica 18 V 125 mm batería 5 Ah', 'category': 'Herramientas eléctricas', 'brand': 'Makita', 'price_cents': 79900, 'min_stock': 8}, {'sku': 'FER-009-01', 'name': 'Broca helicoidal HSS para metal 3 mm', 'category': 'Accesorios de herramientas', 'brand': 'Bosch', 'price_cents': 690, 'min_stock': 8}, {'sku': 'FER-009-02', 'name': 'Broca helicoidal HSS para metal 4 mm', 'category': 'Accesorios de herramientas', 'brand': 'Bosch', 'price_cents': 790, 'min_stock': 8}, {'sku': 'FER-009-03', 'name': 'Broca helicoidal HSS para metal 5 mm', 'category': 'Accesorios de herramientas', 'brand': 'Bosch', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-009-04', 'name': 'Broca helicoidal HSS para metal 6 mm', 'category': 'Accesorios de herramientas', 'brand': 'Bosch', 'price_cents': 1290, 'min_stock': 8}, {'sku': 'FER-009-05', 'name': 'Broca helicoidal HSS para metal 8 mm', 'category': 'Accesorios de herramientas', 'brand': 'Bosch', 'price_cents': 1790, 'min_stock': 8}, {'sku': 'FER-010-01', 'name': 'Broca para concreto con punta de carburo 5 x 85 mm', 'category': 'Accesorios de herramientas', 'brand': 'Bosch', 'price_cents': 890, 'min_stock': 8}, {'sku': 'FER-010-02', 'name': 'Broca para concreto con punta de carburo 6 x 100 mm', 'category': 'Accesorios de herramientas', 'brand': 'Bosch', 'price_cents': 1090, 'min_stock': 8}, {'sku': 'FER-010-03', 'name': 'Broca para concreto con punta de carburo 8 x 120 mm', 'category': 'Accesorios de herramientas', 'brand': 'Bosch', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-010-04', 'name': 'Broca para concreto con punta de carburo 10 x 120 mm', 'category': 'Accesorios de herramientas', 'brand': 'Bosch', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-010-05', 'name': 'Broca para concreto con punta de carburo 12 x 150 mm', 'category': 'Accesorios de herramientas', 'brand': 'Bosch', 'price_cents': 2590, 'min_stock': 8}, {'sku': 'FER-011-01', 'name': 'Disco de corte delgado para metal 115 x 1 mm', 'category': 'Accesorios de herramientas', 'brand': 'Norton', 'price_cents': 590, 'min_stock': 8}, {'sku': 'FER-011-02', 'name': 'Disco de corte delgado para metal 115 x 1.6 mm', 'category': 'Accesorios de herramientas', 'brand': 'Norton', 'price_cents': 690, 'min_stock': 8}, {'sku': 'FER-011-03', 'name': 'Disco de corte delgado para metal 125 x 1 mm', 'category': 'Accesorios de herramientas', 'brand': 'Norton', 'price_cents': 790, 'min_stock': 8}, {'sku': 'FER-011-04', 'name': 'Disco de corte delgado para metal 180 x 1.6 mm', 'category': 'Accesorios de herramientas', 'brand': 'Norton', 'price_cents': 1090, 'min_stock': 8}, {'sku': 'FER-011-05', 'name': 'Disco de corte delgado para metal 230 x 2 mm', 'category': 'Accesorios de herramientas', 'brand': 'Norton', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-012-01', 'name': 'Disco diamantado segmentado 105 mm', 'category': 'Accesorios de herramientas', 'brand': 'DeWalt', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-012-02', 'name': 'Disco diamantado segmentado 115 mm', 'category': 'Accesorios de herramientas', 'brand': 'DeWalt', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-012-03', 'name': 'Disco diamantado segmentado 125 mm', 'category': 'Accesorios de herramientas', 'brand': 'DeWalt', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-012-04', 'name': 'Disco diamantado segmentado 180 mm', 'category': 'Accesorios de herramientas', 'brand': 'DeWalt', 'price_cents': 6990, 'min_stock': 8}, {'sku': 'FER-012-05', 'name': 'Disco diamantado segmentado 230 mm', 'category': 'Accesorios de herramientas', 'brand': 'DeWalt', 'price_cents': 8990, 'min_stock': 8}, {'sku': 'FER-013-01', 'name': 'Badilejo de acero con mango de madera 5 pulgadas', 'category': 'Construcción', 'brand': 'Truper', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-013-02', 'name': 'Badilejo de acero con mango de madera 6 pulgadas', 'category': 'Construcción', 'brand': 'Truper', 'price_cents': 1790, 'min_stock': 8}, {'sku': 'FER-013-03', 'name': 'Badilejo de acero con mango de madera 7 pulgadas', 'category': 'Construcción', 'brand': 'Truper', 'price_cents': 2090, 'min_stock': 8}, {'sku': 'FER-013-04', 'name': 'Badilejo de acero con mango de madera 8 pulgadas', 'category': 'Construcción', 'brand': 'Truper', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-013-05', 'name': 'Badilejo de acero con mango de madera 10 pulgadas', 'category': 'Construcción', 'brand': 'Truper', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-014-01', 'name': 'Llana dentada para cerámicos diente 3 mm', 'category': 'Construcción', 'brand': 'Rubi', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-014-02', 'name': 'Llana dentada para cerámicos diente 4 mm', 'category': 'Construcción', 'brand': 'Rubi', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-014-03', 'name': 'Llana dentada para cerámicos diente 6 mm', 'category': 'Construcción', 'brand': 'Rubi', 'price_cents': 3490, 'min_stock': 8}, {'sku': 'FER-014-04', 'name': 'Llana dentada para cerámicos diente 8 mm', 'category': 'Construcción', 'brand': 'Rubi', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-014-05', 'name': 'Llana dentada para cerámicos diente 10 mm', 'category': 'Construcción', 'brand': 'Rubi', 'price_cents': 4490, 'min_stock': 8}, {'sku': 'FER-015-01', 'name': 'Balde de obra reforzado 8 litros', 'category': 'Construcción', 'brand': 'Truper', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-015-02', 'name': 'Balde de obra reforzado 10 litros', 'category': 'Construcción', 'brand': 'Truper', 'price_cents': 1290, 'min_stock': 8}, {'sku': 'FER-015-03', 'name': 'Balde de obra reforzado 12 litros', 'category': 'Construcción', 'brand': 'Truper', 'price_cents': 1590, 'min_stock': 8}, {'sku': 'FER-015-04', 'name': 'Balde de obra reforzado 15 litros', 'category': 'Construcción', 'brand': 'Truper', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-015-05', 'name': 'Balde de obra reforzado 20 litros', 'category': 'Construcción', 'brand': 'Truper', 'price_cents': 2690, 'min_stock': 8}, {'sku': 'FER-016-01', 'name': 'Cincel plano de acero templado 150 x 12 mm', 'category': 'Construcción', 'brand': 'Stanley', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-016-02', 'name': 'Cincel plano de acero templado 200 x 16 mm', 'category': 'Construcción', 'brand': 'Stanley', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-016-03', 'name': 'Cincel plano de acero templado 250 x 19 mm', 'category': 'Construcción', 'brand': 'Stanley', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-016-04', 'name': 'Cincel plano de acero templado 300 x 22 mm', 'category': 'Construcción', 'brand': 'Stanley', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-016-05', 'name': 'Cincel plano de acero templado 350 x 25 mm', 'category': 'Construcción', 'brand': 'Stanley', 'price_cents': 3690, 'min_stock': 8}, {'sku': 'FER-017-01', 'name': 'Pintura látex blanco mate para interiores 1 litro', 'category': 'Pinturas y acabados', 'brand': 'Dulux', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-017-02', 'name': 'Pintura látex blanco mate para interiores 2.5 litros', 'category': 'Pinturas y acabados', 'brand': 'Dulux', 'price_cents': 5990, 'min_stock': 8}, {'sku': 'FER-017-03', 'name': 'Pintura látex blanco mate para interiores 4 litros', 'category': 'Pinturas y acabados', 'brand': 'Dulux', 'price_cents': 8990, 'min_stock': 8}, {'sku': 'FER-017-04', 'name': 'Pintura látex blanco mate para interiores 10 litros', 'category': 'Pinturas y acabados', 'brand': 'Dulux', 'price_cents': 17990, 'min_stock': 8}, {'sku': 'FER-017-05', 'name': 'Pintura látex blanco mate para interiores 20 litros', 'category': 'Pinturas y acabados', 'brand': 'Dulux', 'price_cents': 29990, 'min_stock': 8}, {'sku': 'FER-018-01', 'name': 'Brocha de cerdas sintéticas 1 pulgada', 'category': 'Pinturas y acabados', 'brand': 'Stanley', 'price_cents': 690, 'min_stock': 8}, {'sku': 'FER-018-02', 'name': 'Brocha de cerdas sintéticas 1.5 pulgadas', 'category': 'Pinturas y acabados', 'brand': 'Stanley', 'price_cents': 890, 'min_stock': 8}, {'sku': 'FER-018-03', 'name': 'Brocha de cerdas sintéticas 2 pulgadas', 'category': 'Pinturas y acabados', 'brand': 'Stanley', 'price_cents': 1190, 'min_stock': 8}, {'sku': 'FER-018-04', 'name': 'Brocha de cerdas sintéticas 3 pulgadas', 'category': 'Pinturas y acabados', 'brand': 'Stanley', 'price_cents': 1790, 'min_stock': 8}, {'sku': 'FER-018-05', 'name': 'Brocha de cerdas sintéticas 4 pulgadas', 'category': 'Pinturas y acabados', 'brand': 'Stanley', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-019-01', 'name': 'Rodillo de microfibra con mango 4 pulgadas', 'category': 'Pinturas y acabados', 'brand': 'Hamilton', 'price_cents': 1290, 'min_stock': 8}, {'sku': 'FER-019-02', 'name': 'Rodillo de microfibra con mango 6 pulgadas', 'category': 'Pinturas y acabados', 'brand': 'Hamilton', 'price_cents': 1790, 'min_stock': 8}, {'sku': 'FER-019-03', 'name': 'Rodillo de microfibra con mango 7 pulgadas', 'category': 'Pinturas y acabados', 'brand': 'Hamilton', 'price_cents': 2190, 'min_stock': 8}, {'sku': 'FER-019-04', 'name': 'Rodillo de microfibra con mango 9 pulgadas', 'category': 'Pinturas y acabados', 'brand': 'Hamilton', 'price_cents': 2790, 'min_stock': 8}, {'sku': 'FER-019-05', 'name': 'Rodillo de microfibra con mango 12 pulgadas', 'category': 'Pinturas y acabados', 'brand': 'Hamilton', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-020-01', 'name': 'Lija al agua para acabados grano 80', 'category': 'Pinturas y acabados', 'brand': 'Norton', 'price_cents': 250, 'min_stock': 8}, {'sku': 'FER-020-02', 'name': 'Lija al agua para acabados grano 120', 'category': 'Pinturas y acabados', 'brand': 'Norton', 'price_cents': 250, 'min_stock': 8}, {'sku': 'FER-020-03', 'name': 'Lija al agua para acabados grano 220', 'category': 'Pinturas y acabados', 'brand': 'Norton', 'price_cents': 290, 'min_stock': 8}, {'sku': 'FER-020-04', 'name': 'Lija al agua para acabados grano 400', 'category': 'Pinturas y acabados', 'brand': 'Norton', 'price_cents': 350, 'min_stock': 8}, {'sku': 'FER-020-05', 'name': 'Lija al agua para acabados grano 600', 'category': 'Pinturas y acabados', 'brand': 'Norton', 'price_cents': 390, 'min_stock': 8}, {'sku': 'FER-021-01', 'name': 'Válvula esférica de bronce roscada 1/2 pulgada', 'category': 'Gasfitería', 'brand': 'Genebre', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-021-02', 'name': 'Válvula esférica de bronce roscada 3/4 pulgada', 'category': 'Gasfitería', 'brand': 'Genebre', 'price_cents': 3490, 'min_stock': 8}, {'sku': 'FER-021-03', 'name': 'Válvula esférica de bronce roscada 1 pulgada', 'category': 'Gasfitería', 'brand': 'Genebre', 'price_cents': 4990, 'min_stock': 8}, {'sku': 'FER-021-04', 'name': 'Válvula esférica de bronce roscada 1 1/4 pulgadas', 'category': 'Gasfitería', 'brand': 'Genebre', 'price_cents': 6990, 'min_stock': 8}, {'sku': 'FER-021-05', 'name': 'Válvula esférica de bronce roscada 1 1/2 pulgadas', 'category': 'Gasfitería', 'brand': 'Genebre', 'price_cents': 8990, 'min_stock': 8}, {'sku': 'FER-022-01', 'name': 'Codo PVC de presión a 90 grados 1/2 pulgada', 'category': 'Gasfitería', 'brand': 'Pavco', 'price_cents': 150, 'min_stock': 8}, {'sku': 'FER-022-02', 'name': 'Codo PVC de presión a 90 grados 3/4 pulgada', 'category': 'Gasfitería', 'brand': 'Pavco', 'price_cents': 250, 'min_stock': 8}, {'sku': 'FER-022-03', 'name': 'Codo PVC de presión a 90 grados 1 pulgada', 'category': 'Gasfitería', 'brand': 'Pavco', 'price_cents': 390, 'min_stock': 8}, {'sku': 'FER-022-04', 'name': 'Codo PVC de presión a 90 grados 1 1/2 pulgadas', 'category': 'Gasfitería', 'brand': 'Pavco', 'price_cents': 790, 'min_stock': 8}, {'sku': 'FER-022-05', 'name': 'Codo PVC de presión a 90 grados 2 pulgadas', 'category': 'Gasfitería', 'brand': 'Pavco', 'price_cents': 1290, 'min_stock': 8}, {'sku': 'FER-023-01', 'name': 'Llave de paso de latón 1/2 pulgada', 'category': 'Gasfitería', 'brand': 'Genebre', 'price_cents': 2790, 'min_stock': 8}, {'sku': 'FER-023-02', 'name': 'Llave de paso de latón 3/4 pulgada', 'category': 'Gasfitería', 'brand': 'Genebre', 'price_cents': 3690, 'min_stock': 8}, {'sku': 'FER-023-03', 'name': 'Llave de paso de latón 1 pulgada', 'category': 'Gasfitería', 'brand': 'Genebre', 'price_cents': 5490, 'min_stock': 8}, {'sku': 'FER-023-04', 'name': 'Llave de paso de latón 1 1/4 pulgadas', 'category': 'Gasfitería', 'brand': 'Genebre', 'price_cents': 7990, 'min_stock': 8}, {'sku': 'FER-023-05', 'name': 'Llave de paso de latón 1 1/2 pulgadas', 'category': 'Gasfitería', 'brand': 'Genebre', 'price_cents': 9990, 'min_stock': 8}, {'sku': 'FER-024-01', 'name': 'Tubo de abasto flexible de acero 1/2 x 20 cm', 'category': 'Gasfitería', 'brand': 'Coflex', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-024-02', 'name': 'Tubo de abasto flexible de acero 1/2 x 30 cm', 'category': 'Gasfitería', 'brand': 'Coflex', 'price_cents': 1690, 'min_stock': 8}, {'sku': 'FER-024-03', 'name': 'Tubo de abasto flexible de acero 1/2 x 40 cm', 'category': 'Gasfitería', 'brand': 'Coflex', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-024-04', 'name': 'Tubo de abasto flexible de acero 1/2 x 50 cm', 'category': 'Gasfitería', 'brand': 'Coflex', 'price_cents': 2290, 'min_stock': 8}, {'sku': 'FER-024-05', 'name': 'Tubo de abasto flexible de acero 1/2 x 60 cm', 'category': 'Gasfitería', 'brand': 'Coflex', 'price_cents': 2590, 'min_stock': 8}, {'sku': 'FER-025-01', 'name': 'Cable unipolar de cobre negro 1.5 mm²', 'category': 'Electricidad', 'brand': 'Indeco', 'price_cents': 250, 'min_stock': 8}, {'sku': 'FER-025-02', 'name': 'Cable unipolar de cobre negro 2.5 mm²', 'category': 'Electricidad', 'brand': 'Indeco', 'price_cents': 390, 'min_stock': 8}, {'sku': 'FER-025-03', 'name': 'Cable unipolar de cobre negro 4 mm²', 'category': 'Electricidad', 'brand': 'Indeco', 'price_cents': 590, 'min_stock': 8}, {'sku': 'FER-025-04', 'name': 'Cable unipolar de cobre negro 6 mm²', 'category': 'Electricidad', 'brand': 'Indeco', 'price_cents': 890, 'min_stock': 8}, {'sku': 'FER-025-05', 'name': 'Cable unipolar de cobre negro 10 mm²', 'category': 'Electricidad', 'brand': 'Indeco', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-026-01', 'name': 'Interruptor termomagnético bipolar 10 A', 'category': 'Electricidad', 'brand': 'Schneider Electric', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-026-02', 'name': 'Interruptor termomagnético bipolar 16 A', 'category': 'Electricidad', 'brand': 'Schneider Electric', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-026-03', 'name': 'Interruptor termomagnético bipolar 20 A', 'category': 'Electricidad', 'brand': 'Schneider Electric', 'price_cents': 4290, 'min_stock': 8}, {'sku': 'FER-026-04', 'name': 'Interruptor termomagnético bipolar 25 A', 'category': 'Electricidad', 'brand': 'Schneider Electric', 'price_cents': 4490, 'min_stock': 8}, {'sku': 'FER-026-05', 'name': 'Interruptor termomagnético bipolar 32 A', 'category': 'Electricidad', 'brand': 'Schneider Electric', 'price_cents': 4990, 'min_stock': 8}, {'sku': 'FER-027-01', 'name': 'Cinta aislante de PVC de uso eléctrico negro 18 m', 'category': 'Electricidad', 'brand': '3M', 'price_cents': 690, 'min_stock': 8}, {'sku': 'FER-027-02', 'name': 'Cinta aislante de PVC de uso eléctrico azul 18 m', 'category': 'Electricidad', 'brand': '3M', 'price_cents': 690, 'min_stock': 8}, {'sku': 'FER-027-03', 'name': 'Cinta aislante de PVC de uso eléctrico rojo 18 m', 'category': 'Electricidad', 'brand': '3M', 'price_cents': 690, 'min_stock': 8}, {'sku': 'FER-027-04', 'name': 'Cinta aislante de PVC de uso eléctrico blanco 18 m', 'category': 'Electricidad', 'brand': '3M', 'price_cents': 690, 'min_stock': 8}, {'sku': 'FER-027-05', 'name': 'Cinta aislante de PVC de uso eléctrico verde 18 m', 'category': 'Electricidad', 'brand': '3M', 'price_cents': 690, 'min_stock': 8}, {'sku': 'FER-028-01', 'name': 'Tubo corrugado flexible para cableado 16 mm x 25 m', 'category': 'Electricidad', 'brand': 'Pavco', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-028-02', 'name': 'Tubo corrugado flexible para cableado 20 mm x 25 m', 'category': 'Electricidad', 'brand': 'Pavco', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-028-03', 'name': 'Tubo corrugado flexible para cableado 25 mm x 25 m', 'category': 'Electricidad', 'brand': 'Pavco', 'price_cents': 4990, 'min_stock': 8}, {'sku': 'FER-028-04', 'name': 'Tubo corrugado flexible para cableado 32 mm x 25 m', 'category': 'Electricidad', 'brand': 'Pavco', 'price_cents': 7990, 'min_stock': 8}, {'sku': 'FER-028-05', 'name': 'Tubo corrugado flexible para cableado 40 mm x 25 m', 'category': 'Electricidad', 'brand': 'Pavco', 'price_cents': 10990, 'min_stock': 8}, {'sku': 'FER-029-01', 'name': 'Foco LED luz blanca E27 7 W', 'category': 'Iluminación', 'brand': 'Philips', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-029-02', 'name': 'Foco LED luz blanca E27 9 W', 'category': 'Iluminación', 'brand': 'Philips', 'price_cents': 1290, 'min_stock': 8}, {'sku': 'FER-029-03', 'name': 'Foco LED luz blanca E27 11 W', 'category': 'Iluminación', 'brand': 'Philips', 'price_cents': 1590, 'min_stock': 8}, {'sku': 'FER-029-04', 'name': 'Foco LED luz blanca E27 13 W', 'category': 'Iluminación', 'brand': 'Philips', 'price_cents': 1890, 'min_stock': 8}, {'sku': 'FER-029-05', 'name': 'Foco LED luz blanca E27 15 W', 'category': 'Iluminación', 'brand': 'Philips', 'price_cents': 2290, 'min_stock': 8}, {'sku': 'FER-030-01', 'name': 'Reflector LED exterior IP65 10 W', 'category': 'Iluminación', 'brand': 'Ledvance', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-030-02', 'name': 'Reflector LED exterior IP65 20 W', 'category': 'Iluminación', 'brand': 'Ledvance', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-030-03', 'name': 'Reflector LED exterior IP65 30 W', 'category': 'Iluminación', 'brand': 'Ledvance', 'price_cents': 5490, 'min_stock': 8}, {'sku': 'FER-030-04', 'name': 'Reflector LED exterior IP65 50 W', 'category': 'Iluminación', 'brand': 'Ledvance', 'price_cents': 7990, 'min_stock': 8}, {'sku': 'FER-030-05', 'name': 'Reflector LED exterior IP65 100 W', 'category': 'Iluminación', 'brand': 'Ledvance', 'price_cents': 13990, 'min_stock': 8}, {'sku': 'FER-031-01', 'name': 'Panel LED redondo empotrable 6 W', 'category': 'Iluminación', 'brand': 'Philips', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-031-02', 'name': 'Panel LED redondo empotrable 9 W', 'category': 'Iluminación', 'brand': 'Philips', 'price_cents': 2790, 'min_stock': 8}, {'sku': 'FER-031-03', 'name': 'Panel LED redondo empotrable 12 W', 'category': 'Iluminación', 'brand': 'Philips', 'price_cents': 3490, 'min_stock': 8}, {'sku': 'FER-031-04', 'name': 'Panel LED redondo empotrable 18 W', 'category': 'Iluminación', 'brand': 'Philips', 'price_cents': 4490, 'min_stock': 8}, {'sku': 'FER-031-05', 'name': 'Panel LED redondo empotrable 24 W', 'category': 'Iluminación', 'brand': 'Philips', 'price_cents': 5990, 'min_stock': 8}, {'sku': 'FER-032-01', 'name': 'Tira LED luz cálida autoadhesiva 1 metro', 'category': 'Iluminación', 'brand': 'Ledvance', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-032-02', 'name': 'Tira LED luz cálida autoadhesiva 2 metros', 'category': 'Iluminación', 'brand': 'Ledvance', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-032-03', 'name': 'Tira LED luz cálida autoadhesiva 3 metros', 'category': 'Iluminación', 'brand': 'Ledvance', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-032-04', 'name': 'Tira LED luz cálida autoadhesiva 5 metros', 'category': 'Iluminación', 'brand': 'Ledvance', 'price_cents': 5990, 'min_stock': 8}, {'sku': 'FER-032-05', 'name': 'Tira LED luz cálida autoadhesiva 10 metros', 'category': 'Iluminación', 'brand': 'Ledvance', 'price_cents': 9990, 'min_stock': 8}, {'sku': 'FER-033-01', 'name': 'Tornillo para madera zincado caja de 100 3 x 20 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-033-02', 'name': 'Tornillo para madera zincado caja de 100 3.5 x 25 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 1290, 'min_stock': 8}, {'sku': 'FER-033-03', 'name': 'Tornillo para madera zincado caja de 100 4 x 30 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 1590, 'min_stock': 8}, {'sku': 'FER-033-04', 'name': 'Tornillo para madera zincado caja de 100 4 x 40 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-033-05', 'name': 'Tornillo para madera zincado caja de 100 5 x 50 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 2790, 'min_stock': 8}, {'sku': 'FER-034-01', 'name': 'Tarugo de nylon universal bolsa de 50 5 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-034-02', 'name': 'Tarugo de nylon universal bolsa de 50 6 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 1290, 'min_stock': 8}, {'sku': 'FER-034-03', 'name': 'Tarugo de nylon universal bolsa de 50 8 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 1790, 'min_stock': 8}, {'sku': 'FER-034-04', 'name': 'Tarugo de nylon universal bolsa de 50 10 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-034-05', 'name': 'Tarugo de nylon universal bolsa de 50 12 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 3490, 'min_stock': 8}, {'sku': 'FER-035-01', 'name': 'Perno hexagonal zincado con tuerca M6 x 30 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 150, 'min_stock': 8}, {'sku': 'FER-035-02', 'name': 'Perno hexagonal zincado con tuerca M8 x 40 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 250, 'min_stock': 8}, {'sku': 'FER-035-03', 'name': 'Perno hexagonal zincado con tuerca M10 x 50 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 390, 'min_stock': 8}, {'sku': 'FER-035-04', 'name': 'Perno hexagonal zincado con tuerca M12 x 60 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 590, 'min_stock': 8}, {'sku': 'FER-035-05', 'name': 'Perno hexagonal zincado con tuerca M16 x 80 mm', 'category': 'Tornillos y fijaciones', 'brand': 'Fischer', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-036-01', 'name': 'Clavo para madera con cabeza 1 pulgada', 'category': 'Tornillos y fijaciones', 'brand': 'Acero comercial', 'price_cents': 790, 'min_stock': 8}, {'sku': 'FER-036-02', 'name': 'Clavo para madera con cabeza 1.5 pulgadas', 'category': 'Tornillos y fijaciones', 'brand': 'Acero comercial', 'price_cents': 790, 'min_stock': 8}, {'sku': 'FER-036-03', 'name': 'Clavo para madera con cabeza 2 pulgadas', 'category': 'Tornillos y fijaciones', 'brand': 'Acero comercial', 'price_cents': 890, 'min_stock': 8}, {'sku': 'FER-036-04', 'name': 'Clavo para madera con cabeza 2.5 pulgadas', 'category': 'Tornillos y fijaciones', 'brand': 'Acero comercial', 'price_cents': 890, 'min_stock': 8}, {'sku': 'FER-036-05', 'name': 'Clavo para madera con cabeza 3 pulgadas', 'category': 'Tornillos y fijaciones', 'brand': 'Acero comercial', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-037-01', 'name': 'Silicona multiuso en cartucho transparente 280 ml', 'category': 'Adhesivos y sellantes', 'brand': 'Sika', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-037-02', 'name': 'Silicona multiuso en cartucho blanco 280 ml', 'category': 'Adhesivos y sellantes', 'brand': 'Sika', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-037-03', 'name': 'Silicona multiuso en cartucho negro 280 ml', 'category': 'Adhesivos y sellantes', 'brand': 'Sika', 'price_cents': 2290, 'min_stock': 8}, {'sku': 'FER-037-04', 'name': 'Silicona multiuso en cartucho gris 280 ml', 'category': 'Adhesivos y sellantes', 'brand': 'Sika', 'price_cents': 2290, 'min_stock': 8}, {'sku': 'FER-037-05', 'name': 'Silicona multiuso en cartucho beige 280 ml', 'category': 'Adhesivos y sellantes', 'brand': 'Sika', 'price_cents': 2290, 'min_stock': 8}, {'sku': 'FER-038-01', 'name': 'Pegamento de montaje de alta resistencia blanco 250 g', 'category': 'Adhesivos y sellantes', 'brand': 'Pattex', 'price_cents': 1790, 'min_stock': 8}, {'sku': 'FER-038-02', 'name': 'Pegamento de montaje de alta resistencia blanco 300 g', 'category': 'Adhesivos y sellantes', 'brand': 'Pattex', 'price_cents': 2190, 'min_stock': 8}, {'sku': 'FER-038-03', 'name': 'Pegamento de montaje de alta resistencia transparente 290 g', 'category': 'Adhesivos y sellantes', 'brand': 'Pattex', 'price_cents': 2790, 'min_stock': 8}, {'sku': 'FER-038-04', 'name': 'Pegamento de montaje de alta resistencia rápido 310 g', 'category': 'Adhesivos y sellantes', 'brand': 'Pattex', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-038-05', 'name': 'Pegamento de montaje de alta resistencia exterior 350 g', 'category': 'Adhesivos y sellantes', 'brand': 'Pattex', 'price_cents': 3490, 'min_stock': 8}, {'sku': 'FER-039-01', 'name': 'Cinta de enmascarar para pintura 18 mm x 40 m', 'category': 'Adhesivos y sellantes', 'brand': '3M', 'price_cents': 590, 'min_stock': 8}, {'sku': 'FER-039-02', 'name': 'Cinta de enmascarar para pintura 24 mm x 40 m', 'category': 'Adhesivos y sellantes', 'brand': '3M', 'price_cents': 790, 'min_stock': 8}, {'sku': 'FER-039-03', 'name': 'Cinta de enmascarar para pintura 36 mm x 40 m', 'category': 'Adhesivos y sellantes', 'brand': '3M', 'price_cents': 1190, 'min_stock': 8}, {'sku': 'FER-039-04', 'name': 'Cinta de enmascarar para pintura 48 mm x 40 m', 'category': 'Adhesivos y sellantes', 'brand': '3M', 'price_cents': 1590, 'min_stock': 8}, {'sku': 'FER-039-05', 'name': 'Cinta de enmascarar para pintura 72 mm x 40 m', 'category': 'Adhesivos y sellantes', 'brand': '3M', 'price_cents': 2290, 'min_stock': 8}, {'sku': 'FER-040-01', 'name': 'Cola sintética para madera 125 ml', 'category': 'Adhesivos y sellantes', 'brand': 'Titebond', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-040-02', 'name': 'Cola sintética para madera 250 ml', 'category': 'Adhesivos y sellantes', 'brand': 'Titebond', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-040-03', 'name': 'Cola sintética para madera 500 ml', 'category': 'Adhesivos y sellantes', 'brand': 'Titebond', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-040-04', 'name': 'Cola sintética para madera 1 litro', 'category': 'Adhesivos y sellantes', 'brand': 'Titebond', 'price_cents': 6490, 'min_stock': 8}, {'sku': 'FER-040-05', 'name': 'Cola sintética para madera 4 litros', 'category': 'Adhesivos y sellantes', 'brand': 'Titebond', 'price_cents': 19990, 'min_stock': 8}, {'sku': 'FER-041-01', 'name': 'Guantes de trabajo recubiertos de nitrilo talla XS', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-041-02', 'name': 'Guantes de trabajo recubiertos de nitrilo talla S', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-041-03', 'name': 'Guantes de trabajo recubiertos de nitrilo talla M', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-041-04', 'name': 'Guantes de trabajo recubiertos de nitrilo talla L', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-041-05', 'name': 'Guantes de trabajo recubiertos de nitrilo talla XL', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-042-01', 'name': 'Lentes de protección envolventes luna clara', 'category': 'Seguridad industrial', 'brand': '3M', 'price_cents': 1290, 'min_stock': 8}, {'sku': 'FER-042-02', 'name': 'Lentes de protección envolventes luna gris', 'category': 'Seguridad industrial', 'brand': '3M', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-042-03', 'name': 'Lentes de protección envolventes luna ámbar', 'category': 'Seguridad industrial', 'brand': '3M', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-042-04', 'name': 'Lentes de protección envolventes antivaho claro', 'category': 'Seguridad industrial', 'brand': '3M', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-042-05', 'name': 'Lentes de protección envolventes antirayadura gris', 'category': 'Seguridad industrial', 'brand': '3M', 'price_cents': 2290, 'min_stock': 8}, {'sku': 'FER-043-01', 'name': 'Casco de seguridad regulable blanco', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-043-02', 'name': 'Casco de seguridad regulable amarillo', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-043-03', 'name': 'Casco de seguridad regulable azul', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-043-04', 'name': 'Casco de seguridad regulable rojo', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-043-05', 'name': 'Casco de seguridad regulable verde', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-044-01', 'name': 'Chaleco reflectivo de alta visibilidad talla S', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-044-02', 'name': 'Chaleco reflectivo de alta visibilidad talla M', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-044-03', 'name': 'Chaleco reflectivo de alta visibilidad talla L', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-044-04', 'name': 'Chaleco reflectivo de alta visibilidad talla XL', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 1690, 'min_stock': 8}, {'sku': 'FER-044-05', 'name': 'Chaleco reflectivo de alta visibilidad talla XXL', 'category': 'Seguridad industrial', 'brand': 'Portwest', 'price_cents': 1690, 'min_stock': 8}, {'sku': 'FER-045-01', 'name': 'Manguera de jardín reforzada 1/2 x 10 m', 'category': 'Jardinería', 'brand': 'Hozelock', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-045-02', 'name': 'Manguera de jardín reforzada 1/2 x 15 m', 'category': 'Jardinería', 'brand': 'Hozelock', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-045-03', 'name': 'Manguera de jardín reforzada 1/2 x 20 m', 'category': 'Jardinería', 'brand': 'Hozelock', 'price_cents': 4990, 'min_stock': 8}, {'sku': 'FER-045-04', 'name': 'Manguera de jardín reforzada 1/2 x 25 m', 'category': 'Jardinería', 'brand': 'Hozelock', 'price_cents': 5990, 'min_stock': 8}, {'sku': 'FER-045-05', 'name': 'Manguera de jardín reforzada 1/2 x 30 m', 'category': 'Jardinería', 'brand': 'Hozelock', 'price_cents': 6990, 'min_stock': 8}, {'sku': 'FER-046-01', 'name': 'Tijera de podar de corte bypass 17 cm', 'category': 'Jardinería', 'brand': 'Fiskars', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-046-02', 'name': 'Tijera de podar de corte bypass 18 cm', 'category': 'Jardinería', 'brand': 'Fiskars', 'price_cents': 4490, 'min_stock': 8}, {'sku': 'FER-046-03', 'name': 'Tijera de podar de corte bypass 20 cm', 'category': 'Jardinería', 'brand': 'Fiskars', 'price_cents': 5490, 'min_stock': 8}, {'sku': 'FER-046-04', 'name': 'Tijera de podar de corte bypass 21 cm', 'category': 'Jardinería', 'brand': 'Fiskars', 'price_cents': 6990, 'min_stock': 8}, {'sku': 'FER-046-05', 'name': 'Tijera de podar de corte bypass 23 cm', 'category': 'Jardinería', 'brand': 'Fiskars', 'price_cents': 8990, 'min_stock': 8}, {'sku': 'FER-047-01', 'name': 'Pala de jardín con mango largo punta redonda 100 cm', 'category': 'Jardinería', 'brand': 'Truper', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-047-02', 'name': 'Pala de jardín con mango largo punta redonda 120 cm', 'category': 'Jardinería', 'brand': 'Truper', 'price_cents': 4490, 'min_stock': 8}, {'sku': 'FER-047-03', 'name': 'Pala de jardín con mango largo punta cuadrada 100 cm', 'category': 'Jardinería', 'brand': 'Truper', 'price_cents': 4290, 'min_stock': 8}, {'sku': 'FER-047-04', 'name': 'Pala de jardín con mango largo punta cuadrada 120 cm', 'category': 'Jardinería', 'brand': 'Truper', 'price_cents': 4990, 'min_stock': 8}, {'sku': 'FER-047-05', 'name': 'Pala de jardín con mango largo punta jardinera 90 cm', 'category': 'Jardinería', 'brand': 'Truper', 'price_cents': 3490, 'min_stock': 8}, {'sku': 'FER-048-01', 'name': 'Rastrillo de jardín de acero 10 dientes', 'category': 'Jardinería', 'brand': 'Truper', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-048-02', 'name': 'Rastrillo de jardín de acero 12 dientes', 'category': 'Jardinería', 'brand': 'Truper', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-048-03', 'name': 'Rastrillo de jardín de acero 14 dientes', 'category': 'Jardinería', 'brand': 'Truper', 'price_cents': 3490, 'min_stock': 8}, {'sku': 'FER-048-04', 'name': 'Rastrillo de jardín de acero 16 dientes', 'category': 'Jardinería', 'brand': 'Truper', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-048-05', 'name': 'Rastrillo de jardín de acero 18 dientes', 'category': 'Jardinería', 'brand': 'Truper', 'price_cents': 4490, 'min_stock': 8}, {'sku': 'FER-049-01', 'name': 'Candado de latón con arco de acero 20 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Yale', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-049-02', 'name': 'Candado de latón con arco de acero 30 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Yale', 'price_cents': 2290, 'min_stock': 8}, {'sku': 'FER-049-03', 'name': 'Candado de latón con arco de acero 40 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Yale', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-049-04', 'name': 'Candado de latón con arco de acero 50 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Yale', 'price_cents': 4290, 'min_stock': 8}, {'sku': 'FER-049-05', 'name': 'Candado de latón con arco de acero 60 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Yale', 'price_cents': 5990, 'min_stock': 8}, {'sku': 'FER-050-01', 'name': 'Bisagra de acero zincado par 2 pulgadas', 'category': 'Cerrajería y herrajes', 'brand': 'Häfele', 'price_cents': 590, 'min_stock': 8}, {'sku': 'FER-050-02', 'name': 'Bisagra de acero zincado par 2.5 pulgadas', 'category': 'Cerrajería y herrajes', 'brand': 'Häfele', 'price_cents': 790, 'min_stock': 8}, {'sku': 'FER-050-03', 'name': 'Bisagra de acero zincado par 3 pulgadas', 'category': 'Cerrajería y herrajes', 'brand': 'Häfele', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-050-04', 'name': 'Bisagra de acero zincado par 3.5 pulgadas', 'category': 'Cerrajería y herrajes', 'brand': 'Häfele', 'price_cents': 1290, 'min_stock': 8}, {'sku': 'FER-050-05', 'name': 'Bisagra de acero zincado par 4 pulgadas', 'category': 'Cerrajería y herrajes', 'brand': 'Häfele', 'price_cents': 1590, 'min_stock': 8}, {'sku': 'FER-051-01', 'name': 'Cerrojo pasador de acero para puerta 50 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Yale', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-051-02', 'name': 'Cerrojo pasador de acero para puerta 75 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Yale', 'price_cents': 1290, 'min_stock': 8}, {'sku': 'FER-051-03', 'name': 'Cerrojo pasador de acero para puerta 100 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Yale', 'price_cents': 1690, 'min_stock': 8}, {'sku': 'FER-051-04', 'name': 'Cerrojo pasador de acero para puerta 150 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Yale', 'price_cents': 2290, 'min_stock': 8}, {'sku': 'FER-051-05', 'name': 'Cerrojo pasador de acero para puerta 200 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Yale', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-052-01', 'name': 'Tirador de acero inoxidable para mueble 96 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Häfele', 'price_cents': 890, 'min_stock': 8}, {'sku': 'FER-052-02', 'name': 'Tirador de acero inoxidable para mueble 128 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Häfele', 'price_cents': 1190, 'min_stock': 8}, {'sku': 'FER-052-03', 'name': 'Tirador de acero inoxidable para mueble 160 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Häfele', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-052-04', 'name': 'Tirador de acero inoxidable para mueble 192 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Häfele', 'price_cents': 1890, 'min_stock': 8}, {'sku': 'FER-052-05', 'name': 'Tirador de acero inoxidable para mueble 256 mm', 'category': 'Cerrajería y herrajes', 'brand': 'Häfele', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-053-01', 'name': 'Escoba industrial de cerdas duras 30 cm', 'category': 'Limpieza y mantenimiento', 'brand': 'Truper', 'price_cents': 1790, 'min_stock': 8}, {'sku': 'FER-053-02', 'name': 'Escoba industrial de cerdas duras 40 cm', 'category': 'Limpieza y mantenimiento', 'brand': 'Truper', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-053-03', 'name': 'Escoba industrial de cerdas duras 50 cm', 'category': 'Limpieza y mantenimiento', 'brand': 'Truper', 'price_cents': 3290, 'min_stock': 8}, {'sku': 'FER-053-04', 'name': 'Escoba industrial de cerdas duras 60 cm', 'category': 'Limpieza y mantenimiento', 'brand': 'Truper', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-053-05', 'name': 'Escoba industrial de cerdas duras 80 cm', 'category': 'Limpieza y mantenimiento', 'brand': 'Truper', 'price_cents': 5990, 'min_stock': 8}, {'sku': 'FER-054-01', 'name': 'Paño de microfibra multiuso paquete 2 unidades', 'category': 'Limpieza y mantenimiento', 'brand': 'Vileda', 'price_cents': 690, 'min_stock': 8}, {'sku': 'FER-054-02', 'name': 'Paño de microfibra multiuso paquete 3 unidades', 'category': 'Limpieza y mantenimiento', 'brand': 'Vileda', 'price_cents': 990, 'min_stock': 8}, {'sku': 'FER-054-03', 'name': 'Paño de microfibra multiuso paquete 5 unidades', 'category': 'Limpieza y mantenimiento', 'brand': 'Vileda', 'price_cents': 1590, 'min_stock': 8}, {'sku': 'FER-054-04', 'name': 'Paño de microfibra multiuso paquete 8 unidades', 'category': 'Limpieza y mantenimiento', 'brand': 'Vileda', 'price_cents': 2390, 'min_stock': 8}, {'sku': 'FER-054-05', 'name': 'Paño de microfibra multiuso paquete 10 unidades', 'category': 'Limpieza y mantenimiento', 'brand': 'Vileda', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-055-01', 'name': 'Cepillo de alambre de acero 3 hileras', 'category': 'Limpieza y mantenimiento', 'brand': 'Stanley', 'price_cents': 690, 'min_stock': 8}, {'sku': 'FER-055-02', 'name': 'Cepillo de alambre de acero 4 hileras', 'category': 'Limpieza y mantenimiento', 'brand': 'Stanley', 'price_cents': 890, 'min_stock': 8}, {'sku': 'FER-055-03', 'name': 'Cepillo de alambre de acero 5 hileras', 'category': 'Limpieza y mantenimiento', 'brand': 'Stanley', 'price_cents': 1090, 'min_stock': 8}, {'sku': 'FER-055-04', 'name': 'Cepillo de alambre de acero 6 hileras', 'category': 'Limpieza y mantenimiento', 'brand': 'Stanley', 'price_cents': 1390, 'min_stock': 8}, {'sku': 'FER-055-05', 'name': 'Cepillo de alambre de acero 7 hileras', 'category': 'Limpieza y mantenimiento', 'brand': 'Stanley', 'price_cents': 1690, 'min_stock': 8}, {'sku': 'FER-056-01', 'name': 'Bolsa para residuos de obra reforzada paquete 10 unidades 40 L', 'category': 'Limpieza y mantenimiento', 'brand': 'Marca propia', 'price_cents': 1290, 'min_stock': 8}, {'sku': 'FER-056-02', 'name': 'Bolsa para residuos de obra reforzada paquete 10 unidades 60 L', 'category': 'Limpieza y mantenimiento', 'brand': 'Marca propia', 'price_cents': 1790, 'min_stock': 8}, {'sku': 'FER-056-03', 'name': 'Bolsa para residuos de obra reforzada paquete 10 unidades 80 L', 'category': 'Limpieza y mantenimiento', 'brand': 'Marca propia', 'price_cents': 2290, 'min_stock': 8}, {'sku': 'FER-056-04', 'name': 'Bolsa para residuos de obra reforzada paquete 10 unidades 100 L', 'category': 'Limpieza y mantenimiento', 'brand': 'Marca propia', 'price_cents': 2790, 'min_stock': 8}, {'sku': 'FER-056-05', 'name': 'Bolsa para residuos de obra reforzada paquete 10 unidades 120 L', 'category': 'Limpieza y mantenimiento', 'brand': 'Marca propia', 'price_cents': 3290, 'min_stock': 8}, {'sku': 'FER-057-01', 'name': 'Caja de herramientas plástica con bandeja 12 pulgadas', 'category': 'Organización y almacenamiento', 'brand': 'Stanley', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-057-02', 'name': 'Caja de herramientas plástica con bandeja 16 pulgadas', 'category': 'Organización y almacenamiento', 'brand': 'Stanley', 'price_cents': 4990, 'min_stock': 8}, {'sku': 'FER-057-03', 'name': 'Caja de herramientas plástica con bandeja 19 pulgadas', 'category': 'Organización y almacenamiento', 'brand': 'Stanley', 'price_cents': 6990, 'min_stock': 8}, {'sku': 'FER-057-04', 'name': 'Caja de herramientas plástica con bandeja 22 pulgadas', 'category': 'Organización y almacenamiento', 'brand': 'Stanley', 'price_cents': 10990, 'min_stock': 8}, {'sku': 'FER-057-05', 'name': 'Caja de herramientas plástica con bandeja 26 pulgadas', 'category': 'Organización y almacenamiento', 'brand': 'Stanley', 'price_cents': 15990, 'min_stock': 8}, {'sku': 'FER-058-01', 'name': 'Organizador de piezas con divisiones 8 compartimentos', 'category': 'Organización y almacenamiento', 'brand': 'Stanley', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-058-02', 'name': 'Organizador de piezas con divisiones 10 compartimentos', 'category': 'Organización y almacenamiento', 'brand': 'Stanley', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-058-03', 'name': 'Organizador de piezas con divisiones 12 compartimentos', 'category': 'Organización y almacenamiento', 'brand': 'Stanley', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-058-04', 'name': 'Organizador de piezas con divisiones 16 compartimentos', 'category': 'Organización y almacenamiento', 'brand': 'Stanley', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-058-05', 'name': 'Organizador de piezas con divisiones 20 compartimentos', 'category': 'Organización y almacenamiento', 'brand': 'Stanley', 'price_cents': 4990, 'min_stock': 8}, {'sku': 'FER-059-01', 'name': 'Maletín de lona para herramientas 12 pulgadas', 'category': 'Organización y almacenamiento', 'brand': 'DeWalt', 'price_cents': 5990, 'min_stock': 8}, {'sku': 'FER-059-02', 'name': 'Maletín de lona para herramientas 14 pulgadas', 'category': 'Organización y almacenamiento', 'brand': 'DeWalt', 'price_cents': 7990, 'min_stock': 8}, {'sku': 'FER-059-03', 'name': 'Maletín de lona para herramientas 16 pulgadas', 'category': 'Organización y almacenamiento', 'brand': 'DeWalt', 'price_cents': 9990, 'min_stock': 8}, {'sku': 'FER-059-04', 'name': 'Maletín de lona para herramientas 18 pulgadas', 'category': 'Organización y almacenamiento', 'brand': 'DeWalt', 'price_cents': 12990, 'min_stock': 8}, {'sku': 'FER-059-05', 'name': 'Maletín de lona para herramientas 20 pulgadas', 'category': 'Organización y almacenamiento', 'brand': 'DeWalt', 'price_cents': 15990, 'min_stock': 8}, {'sku': 'FER-060-01', 'name': 'Escalera de tijera de aluminio 3 peldaños', 'category': 'Organización y almacenamiento', 'brand': 'Werner', 'price_cents': 14990, 'min_stock': 8}, {'sku': 'FER-060-02', 'name': 'Escalera de tijera de aluminio 4 peldaños', 'category': 'Organización y almacenamiento', 'brand': 'Werner', 'price_cents': 19990, 'min_stock': 8}, {'sku': 'FER-060-03', 'name': 'Escalera de tijera de aluminio 5 peldaños', 'category': 'Organización y almacenamiento', 'brand': 'Werner', 'price_cents': 24990, 'min_stock': 8}, {'sku': 'FER-060-04', 'name': 'Escalera de tijera de aluminio 6 peldaños', 'category': 'Organización y almacenamiento', 'brand': 'Werner', 'price_cents': 29990, 'min_stock': 8}, {'sku': 'FER-060-05', 'name': 'Escalera de tijera de aluminio 8 peldaños', 'category': 'Organización y almacenamiento', 'brand': 'Werner', 'price_cents': 37990, 'min_stock': 8}, {'sku': 'FER-061-01', 'name': 'Wincha métrica con bloqueo 3 metros', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 1490, 'min_stock': 8}, {'sku': 'FER-061-02', 'name': 'Wincha métrica con bloqueo 5 metros', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 2290, 'min_stock': 8}, {'sku': 'FER-061-03', 'name': 'Wincha métrica con bloqueo 7.5 metros', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-061-04', 'name': 'Wincha métrica con bloqueo 8 metros', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 4490, 'min_stock': 8}, {'sku': 'FER-061-05', 'name': 'Wincha métrica con bloqueo 10 metros', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 5990, 'min_stock': 8}, {'sku': 'FER-062-01', 'name': 'Nivel de burbuja de aluminio 30 cm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 2990, 'min_stock': 8}, {'sku': 'FER-062-02', 'name': 'Nivel de burbuja de aluminio 40 cm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 3990, 'min_stock': 8}, {'sku': 'FER-062-03', 'name': 'Nivel de burbuja de aluminio 60 cm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 4990, 'min_stock': 8}, {'sku': 'FER-062-04', 'name': 'Nivel de burbuja de aluminio 80 cm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 6490, 'min_stock': 8}, {'sku': 'FER-062-05', 'name': 'Nivel de burbuja de aluminio 120 cm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 8990, 'min_stock': 8}, {'sku': 'FER-063-01', 'name': 'Escuadra de carpintero de acero 150 mm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 1790, 'min_stock': 8}, {'sku': 'FER-063-02', 'name': 'Escuadra de carpintero de acero 200 mm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 2290, 'min_stock': 8}, {'sku': 'FER-063-03', 'name': 'Escuadra de carpintero de acero 250 mm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 2790, 'min_stock': 8}, {'sku': 'FER-063-04', 'name': 'Escuadra de carpintero de acero 300 mm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 3290, 'min_stock': 8}, {'sku': 'FER-063-05', 'name': 'Escuadra de carpintero de acero 400 mm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 4490, 'min_stock': 8}, {'sku': 'FER-064-01', 'name': 'Regla de acero inoxidable graduada 150 mm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 790, 'min_stock': 8}, {'sku': 'FER-064-02', 'name': 'Regla de acero inoxidable graduada 300 mm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 1290, 'min_stock': 8}, {'sku': 'FER-064-03', 'name': 'Regla de acero inoxidable graduada 500 mm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 1990, 'min_stock': 8}, {'sku': 'FER-064-04', 'name': 'Regla de acero inoxidable graduada 600 mm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 2490, 'min_stock': 8}, {'sku': 'FER-064-05', 'name': 'Regla de acero inoxidable graduada 1000 mm', 'category': 'Medición y trazado', 'brand': 'Stanley', 'price_cents': 3990, 'min_stock': 8}]
cat = pd.DataFrame(CATALOGO)
assert len(cat) == 320 and cat.sku.is_unique
print(f'{len(cat)} productos; {cat.category.nunique()} categorías.')
display(cat.head())

## 2. Dataset reproducible
Una fila por fecha y SKU. Simulamos diferencias de rotación entre productos,
efecto semanal, estacionalidad suave y promociones. La semilla permite repetirlo.
`stock_escenario` representa existencias hipotéticas al cierre: **no limita las ventas
generadas ni se usa para aprender ventas**. Nunca se usan las ventas futuras como entrada.

Para una aplicación real, sustituye esta simulación por ventas diarias, promociones,
devoluciones y stock histórico de tu negocio. Las ventas perdidas por falta de stock
necesitan tratamiento específico: ventas observadas y demanda no son equivalentes.

In [ ]:
rng = np.random.default_rng(SEED)
fechas = pd.date_range(INICIO, periods=DIAS, freq='D')
partes = []
for p in CATALOGO:
    precio = p['price_cents'] / 100
    base = np.clip(7 / (1 + precio / 90) * rng.lognormal(0, .55), .5, 22)
    fase = rng.uniform(0, 2*np.pi)
    promo = (rng.random(DIAS) < .12).astype(int)
    semanal = np.array([1., 1.04, 1.08, 1.06, 1.2, 1.45, .72])[fechas.dayofweek]
    estacional = 1 + .18*np.sin(2*np.pi*np.arange(DIAS)/365 + fase)
    intensidad = base * semanal * estacional * (1 + .35*promo)
    ventas = rng.poisson(intensidad)
    # Escenarios de cobertura variados, independientes de las ventas futuras.
    cobertura = rng.uniform(.25, 2.4, DIAS)
    stock = np.maximum(0, np.rint(base * 7 * cobertura)).astype(int)
    partes.append(pd.DataFrame({
        'fecha': fechas, 'sku': p['sku'], 'producto': p['name'],
        'categoria': p['category'], 'marca': p['brand'], 'precio_soles': precio,
        'promocion': promo, 'unidades_vendidas': ventas, 'stock_escenario': stock,
        'stock_minimo': p['min_stock'], 'origen': 'SINTETICO'
    }))
datos = pd.concat(partes, ignore_index=True).sort_values(['sku', 'fecha']).reset_index(drop=True)
assert len(datos) == 116800
assert not datos.duplicated(['sku', 'fecha']).any()
assert datos.groupby('sku').size().eq(DIAS).all()
assert datos['unidades_vendidas'].ge(0).all()
datos.to_csv(SALIDA / 'dataset_116800.csv', index=False, encoding='utf-8-sig')
print(f'{len(datos):,} registros | {datos.fecha.min().date()} a {datos.fecha.max().date()}')
display(datos.head())

## 3. Variables conocidas al cierre y objetivo futuro
Se pronostica al **cierre del día t**: podemos conocer las ventas de t y los 27 días
anteriores. El objetivo suma t+1 hasta t+7. Las promociones futuras no se conocen.
Eliminamos 27 días iniciales sin historial completo y 7 finales sin objetivo completo.
Las filas de esos últimos días sí se conservan para la predicción final.

In [ ]:
panel = datos.copy()
g = panel.groupby('sku', sort=False)['unidades_vendidas']
panel['ventas_hoy'] = panel['unidades_vendidas']
for lag in [1, 7, 14]:
    panel[f'lag_{lag}'] = g.shift(lag)
for ventana in [7, 14, 28]:
    panel[f'media_{ventana}'] = g.transform(lambda s: s.rolling(ventana).mean())
panel['desv_28'] = g.transform(lambda s: s.rolling(28).std())
panel['dia_semana'] = panel.fecha.dt.dayofweek
panel['mes'] = panel.fecha.dt.month
panel['dia_anio_sin'] = np.sin(2*np.pi*panel.fecha.dt.dayofyear/365.25)
panel['dia_anio_cos'] = np.cos(2*np.pi*panel.fecha.dt.dayofyear/365.25)
panel['objetivo_7d'] = sum(g.shift(-i) for i in range(1, HORIZONTE+1))
CATEGORICAS = ['categoria', 'marca']
NUMERICAS = ['precio_soles', 'promocion', 'ventas_hoy', 'lag_1', 'lag_7', 'lag_14',
             'media_7', 'media_14', 'media_28', 'desv_28', 'dia_semana', 'mes',
             'dia_anio_sin', 'dia_anio_cos']
FEATURES = CATEGORICAS + NUMERICAS
usable = panel.dropna(subset=FEATURES + ['objetivo_7d']).copy()
assert len(usable) == 320*(365-27-7)
assert not any(c in FEATURES for c in ['objetivo_7d', 'stock_escenario'])
print('Filas con historial y etiqueta completos:', len(usable))

## 4. Separación cronológica sin fuga del objetivo
Entrenamiento: orígenes de pronóstico antes de 2026-05-11.
Validación: 2026-05-18 a 2026-07-17.
Prueba final: 2026-07-25 a 2026-10-02.
Los huecos de 7 días impiden que las etiquetas de una partición entren en la siguiente.
El historial pasado para calcular variables sí puede cruzar una frontera: ya sería
conocido al hacer una predicción diaria. La prueba es diaria, con historial actualizado,
no un pronóstico de tres meses generado de una sola vez.

Seleccionamos parámetros y política de alertas únicamente con validación.
La prueba se evalúa una vez. Véase el ejemplo oficial de
[validación temporal con variables históricas](https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html).

In [ ]:
inicio_val = pd.Timestamp('2026-05-18')
inicio_test = pd.Timestamp('2026-07-25')
train = usable[usable.fecha < inicio_val-pd.Timedelta(days=HORIZONTE)].copy()
val = usable[(usable.fecha >= inicio_val) & (usable.fecha < inicio_test-pd.Timedelta(days=HORIZONTE))].copy()
test = usable[usable.fecha >= inicio_test].copy()
assert train.fecha.max()+pd.Timedelta(days=HORIZONTE) < val.fecha.min()
assert val.fecha.max()+pd.Timedelta(days=HORIZONTE) < test.fecha.min()
assert all(len(x) > 0 for x in [train, val, test])
particiones = pd.DataFrame([{'particion': nombre, 'filas': len(d),
    'desde': str(d.fecha.min().date()), 'hasta': str(d.fecha.max().date())}
    for nombre, d in [('Entrenamiento', train), ('Validación', val), ('Prueba', test)]])
display(particiones)
particiones.to_csv(SALIDA/'particiones.csv', index=False)

def crear_modelo(parametros):
    prep = ColumnTransformer([
        ('categorias', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan), CATEGORICAS),
        ('numeros', 'passthrough', NUMERICAS)
    ])
    return make_pipeline(prep, HistGradientBoostingRegressor(
        loss='poisson', early_stopping=False, random_state=SEED,
        categorical_features=[True, True] + [False]*len(NUMERICAS),
        learning_rate=.08, l2_regularization=2, **parametros))

def predecir(modelo, d):
    with threadpool_limits(limits=2):
        return np.maximum(0, modelo.predict(d[FEATURES]))

def metricas(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    denominador = np.abs(y).sum()
    return {'MAE_unidades': float(mean_absolute_error(y, pred)),
        'RMSE_unidades': float(np.sqrt(mean_squared_error(y, pred))),
        'WAPE_pct': float(100*np.abs(y-pred).sum()/denominador) if denominador else np.nan,
        'R2': float(r2_score(y, pred))}

candidatos = [dict(max_iter=100, max_leaf_nodes=15), dict(max_iter=180, max_leaf_nodes=31)]
modelos, resultados_val = [], []
for params in candidatos:
    m = crear_modelo(params)
    with threadpool_limits(limits=2):
        m.fit(train[FEATURES], train.objetivo_7d)
    met = metricas(val.objetivo_7d, predecir(m, val))
    modelos.append(m)
    resultados_val.append({**params, **met})
seleccion = int(np.argmin([r['MAE_unidades'] for r in resultados_val]))
PARAMETROS = candidatos[seleccion]
modelo_evaluado = modelos[seleccion]
display(pd.DataFrame(resultados_val))
print('Parámetros elegidos por MAE de validación:', PARAMETROS)

## 5. Métricas del modelo frente a una referencia
Referencia: repetir el volumen vendido en los 7 días anteriores (`media_7 × 7`).
**MAE:** error absoluto medio, en unidades por producto y semana.
**RMSE:** penaliza más los errores grandes, mismas unidades.
**WAPE:** error absoluto total dividido entre ventas totales, en porcentaje.
**R²:** capacidad de explicar la variación; puede ser negativo.
No usamos MAPE por fila, ya que puede dividir entre cero cuando no hay ventas.
Los objetivos de ventanas consecutivas se solapan: no son muestras independientes.

In [ ]:
pred_test = predecir(modelo_evaluado, test)
pred_val = predecir(modelo_evaluado, val)
baseline_test = test.media_7.to_numpy()*HORIZONTE
comparacion = pd.DataFrame([
    {'modelo': 'Referencia: ventas últimos 7 días', **metricas(test.objetivo_7d, baseline_test)},
    {'modelo': 'HistGradientBoosting', **metricas(test.objetivo_7d, pred_test)}
])
display(comparacion.round(4))
comparacion.to_csv(SALIDA/'metricas_regresion.csv', index=False, encoding='utf-8-sig')
evaluacion = test[['fecha', 'sku', 'producto', 'categoria', 'objetivo_7d', 'stock_escenario']].copy()
evaluacion['prediccion_7d'] = pred_test
evaluacion['referencia_7d'] = baseline_test
por_categoria = pd.DataFrame([{'categoria': nombre, **metricas(d.objetivo_7d, d.prediccion_7d)}
    for nombre, d in evaluacion.groupby('categoria')])
display(por_categoria.round(3))
por_categoria.to_csv(SALIDA/'metricas_por_categoria.csv', index=False, encoding='utf-8-sig')

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(test.objetivo_7d, pred_test, s=4, alpha=.12)
lim = max(test.objetivo_7d.max(), pred_test.max())
ax[0].plot([0, lim], [0, lim], color='crimson')
ax[0].set(xlabel='Ventas reales simuladas (7 días)', ylabel='Predicción (7 días)', title='Prueba: observado vs predicho')
ax[1].hist(pred_test-test.objetivo_7d, bins=60, color='#167d91')
ax[1].set(xlabel='Predicción − observado, unidades', ylabel='Frecuencia', title='Distribución de errores')
fig.tight_layout(); fig.savefig(SALIDA/'errores_modelo.png', dpi=160); plt.show()

## 6. Matriz de confusión y métricas de alertas
La regresión estima una cantidad, por lo que su evaluación principal es MAE/RMSE/WAPE/R².
Para obtener una matriz de confusión, derivamos una decisión binaria:
**riesgo real simulado = ventas de los próximos 7 días > stock del escenario**.
Alerta prevista = predicción × factor de seguridad > stock.

Probamos factores 1.00, 1.10, 1.20 y 1.30 en validación. Elegimos el que reduce el
coste de errores: un riesgo omitido vale 3 puntos y una falsa alarma 1 punto.
Estos son pesos ilustrativos, **no soles ni costes comprobados**. Una falsa alarma
es una reposición innecesaria; un falso negativo es un riesgo de faltante no detectado.
El factor de seguridad no es un intervalo de confianza ni promete un nivel de servicio.

Métricas: exactitud, precisión, sensibilidad (recall), F1 y exactitud balanceada.
La [matriz de confusión](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
usa filas para la clase real y columnas para la predicha.

In [ ]:
riesgo_val = (val.objetivo_7d.to_numpy() > val.stock_escenario.to_numpy()).astype(int)
politicas = []
for factor in [1., 1.1, 1.2, 1.3]:
    alerta = (pred_val*factor > val.stock_escenario.to_numpy()).astype(int)
    tn, fp, fn, tp = confusion_matrix(riesgo_val, alerta, labels=[0, 1]).ravel()
    politicas.append({'factor': factor, 'falsas_alarmas': int(fp), 'riesgos_omitidos': int(fn),
                     'coste_puntos': int(fp+3*fn)})
politicas = pd.DataFrame(politicas)
FACTOR = float(politicas.loc[politicas.coste_puntos.idxmin(), 'factor'])
display(politicas)
politicas.to_csv(SALIDA/'politicas_validacion.csv', index=False)

real = (test.objetivo_7d.to_numpy() > test.stock_escenario.to_numpy()).astype(int)
alerta = (pred_test*FACTOR > test.stock_escenario.to_numpy()).astype(int)
cm = confusion_matrix(real, alerta, labels=[0, 1])
met_alertas = {'factor_validado': FACTOR, 'exactitud': float(accuracy_score(real, alerta)),
    'precision': float(precision_score(real, alerta, zero_division=0)),
    'recall': float(recall_score(real, alerta, zero_division=0)),
    'F1': float(f1_score(real, alerta, zero_division=0)),
    'exactitud_balanceada': float(balanced_accuracy_score(real, alerta)),
    'casos_sin_riesgo': int((real == 0).sum()), 'casos_con_riesgo': int((real == 1).sum())}
display(pd.DataFrame([met_alertas]).round(4))
pd.DataFrame([met_alertas]).to_csv(SALIDA/'metricas_alertas.csv', index=False)
pd.DataFrame(cm, index=['Real: sin riesgo', 'Real: riesgo'],
    columns=['Predicho: sin alerta', 'Predicho: alerta']).to_csv(SALIDA/'matriz_confusion.csv', encoding='utf-8-sig')
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=['Sin riesgo', 'Riesgo']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_xlabel('Clase predicha'); ax.set_ylabel('Clase real')
ax.set_title('Alertas de inventario · prueba sintética'); fig.tight_layout()
fig.savefig(SALIDA/'matriz_confusion.png', dpi=170); plt.show()
evaluacion['riesgo_real'] = real
evaluacion['alerta_predicha'] = alerta
evaluacion.to_csv(SALIDA/'predicciones_prueba.csv', index=False, encoding='utf-8-sig')

## 7. Matriz de decisión de inventario
Esta es la matriz de reglas solicitada. Es distinta de la matriz de confusión.
Asumimos 7 días de cobertura y ningún pedido en tránsito. Para otro plazo de entrega,
hay que ajustar el horizonte, los compromisos, las recepciones y el stock de seguridad.

| Estado al cierre | Decisión | Cantidad sugerida |
|---|---|---|
| Stock inferior a ventas previstas | Reponer urgente | Objetivo − stock |
| Stock cubre previsión, pero no objetivo con reserva | Reponer preventivo | Objetivo − stock |
| Stock cubre el objetivo | Mantener | 0 |

Objetivo = máximo entre stock mínimo de catálogo y previsión × factor validado.
Se redondea hacia arriba a unidades enteras. No es una orden de compra automática.

In [ ]:
# Modelo para uso futuro: reentrenado con todos los objetivos ya conocidos.
# Sus métricas NO se calculan sobre datos usados en este reajuste.
modelo_final = crear_modelo(PARAMETROS)
with threadpool_limits(limits=2):
    modelo_final.fit(usable[FEATURES], usable.objetivo_7d)
ultimo = panel[panel.fecha == panel.fecha.max()].copy()
assert len(ultimo) == 320 and ultimo[FEATURES].notna().all().all()
decision = ultimo[['fecha', 'sku', 'producto', 'categoria', 'precio_soles', 'stock_escenario', 'stock_minimo']].copy()
decision['ventas_previstas_7d'] = predecir(modelo_final, ultimo)
decision['objetivo_stock'] = np.ceil(np.maximum(decision.stock_minimo, decision.ventas_previstas_7d*FACTOR)).astype(int)
decision['cantidad_sugerida'] = np.maximum(0, decision.objetivo_stock-decision.stock_escenario).astype(int)
decision['decision'] = np.select([
    decision.stock_escenario < decision.ventas_previstas_7d,
    decision.stock_escenario < decision.objetivo_stock
], ['Reponer urgente', 'Reponer preventivo'], default='Mantener')
decision['valor_catalogo_soles'] = (decision.cantidad_sugerida*decision.precio_soles).round(2)
# Valor referencial al precio de venta; no equivale a coste de compra.
decision = decision.sort_values('cantidad_sugerida', ascending=False)
display(decision.head(20).round({'ventas_previstas_7d': 2}))
decision.to_csv(SALIDA/'matriz_decision_320_productos.csv', index=False, encoding='utf-8-sig')
resumen_decision = pd.crosstab(decision.categoria, decision.decision).reindex(
    columns=['Reponer urgente', 'Reponer preventivo', 'Mantener'], fill_value=0)
display(resumen_decision)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(resumen_decision, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(3), resumen_decision.columns)
ax.set_yticks(range(len(resumen_decision)), resumen_decision.index)
for i in range(len(resumen_decision)):
    for j in range(3):
        ax.text(j, i, str(resumen_decision.iloc[i,j]), ha='center', va='center', color='black')
ax.set_title('Matriz de decisión · número de productos por categoría')
fig.colorbar(im, ax=ax, label='Productos'); fig.tight_layout()
fig.savefig(SALIDA/'matriz_decision.png', dpi=170); plt.show()

## 8. Guardar y descargar todo
Se exportan el dataset, métricas, predicciones de prueba, dos matrices, gráficos y modelo.
`modelo_evaluado.joblib` corresponde a las métricas de prueba. `modelo_final.joblib`
se reajusta con todos los datos etiquetados y sirve para nuevos pronósticos.
El modelo final se guarda junto a columnas y política de decisión para reutilizarlo.
Los archivos joblib solo deben cargarse si proceden de una fuente de confianza.

**Recomendación:** primero usa este experimento para aprender el flujo; después vuelve a
entrenar con el historial real. Pasar de 116,800 a 233,600 filas sintéticas no acredita
mayor calidad. Más meses reales, calidad de stock y una buena referencia sí permiten
evaluar si el modelo aporta valor. La aplicación actual no cambia con este cuaderno.

In [ ]:
joblib.dump({'modelo': modelo_evaluado, 'features': FEATURES,
    'factor': FACTOR, 'horizonte_dias': HORIZONTE}, SALIDA/'modelo_evaluado.joblib')
joblib.dump({'modelo': modelo_final, 'features': FEATURES,
    'factor': FACTOR, 'horizonte_dias': HORIZONTE}, SALIDA/'modelo_final.joblib')
informe = {'origen': 'SINTETICO', 'semilla': SEED, 'filas_dataset': len(datos),
    'productos': len(CATALOGO), 'dias': DIAS, 'horizonte': HORIZONTE,
    'parametros': PARAMETROS, 'regresion_prueba': comparacion.to_dict(orient='records'),
    'alertas_prueba': met_alertas, 'matriz_confusion': cm.tolist(),
    'particiones': particiones.to_dict(orient='records'),
    'versiones': {'python': platform.python_version(), 'numpy': np.__version__,
        'pandas': pd.__version__, 'sklearn': sklearn.__version__},
    'supuestos': ['Ventas sin restricciones de stock', 'Stock hipotético al cierre',
        'Sin recepciones en los próximos 7 días', 'No constituye validación comercial']}
(SALIDA/'informe.json').write_text(json.dumps(informe, ensure_ascii=False, indent=2), encoding='utf-8')
diccionario = {
    'fecha': 'Fecha diaria, calendario de Perú; pronóstico al cierre del día.',
    'sku': 'Identificador único del producto.', 'producto': 'Nombre de catálogo.',
    'categoria': 'Categoría de catálogo.', 'marca': 'Marca de catálogo.',
    'precio_soles': 'Precio de catálogo de ejemplo en PEN, no precio actual verificado.',
    'promocion': '0/1: promoción simulada del día.',
    'unidades_vendidas': 'Ventas diarias simuladas, sin límite de stock.',
    'stock_escenario': 'Stock hipotético al cierre para evaluar alertas; no limita ventas.',
    'stock_minimo': 'Mínimo de catálogo para ejemplo de política.',
    'origen': 'SINTETICO en todas las filas.'}
(SALIDA/'diccionario_datos.json').write_text(json.dumps(diccionario, ensure_ascii=False, indent=2), encoding='utf-8')
archivo = Path('Horizonte_Colab_resultados.zip')
with zipfile.ZipFile(archivo, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(SALIDA.iterdir()):
        if f.is_file(): z.write(f, arcname=f.name)
print('LISTO:', archivo.resolve(), '|', round(archivo.stat().st_size/1024/1024, 2), 'MB')
try:
    from google.colab import files
except ImportError:
    print('Ejecución local: abre el ZIP en la carpeta del cuaderno.')
else:
    files.download(str(archivo))